# Week-13: Classification Task in Python

<font size='4'>

* Welcome back!
* Today, we will go over a classification example using `skicit-learn` package in Python.

<font size='4'>

* HW8 Q1: The location of files are misplaced. Suppose that `python_proj` is your working directory, <br> `HW8_main.py` should be put right under `python_proj`, while `HW8Fun.py` should be put under `self_py_fun` folder.
* Interpretation of parameters of the linear model is not correct:
* <img src="figures/HW8_linear_model_output.png" alt="drawing" width="500"/>
* For continuous covariate, i.e., CTQ total score, you should write:
    * One score increase in CTQ total score, on average, corresponds to a 0.042 score increase in PCL5 score at intake, adjusting for other covariates (p=0.025, 95% CI: [0.003, 0.081]).
* For categorical covariate, i.e., military rank, you should have reference group:
    * Patients who are officers, on average, had lower PCL5 score at intake by 3.06 than patients who are enlisted, adjusting for other covariates (p=0.022, 95% CI: [0.44, 5.67]). 

## Import data

<font size='4'>

* `Scikit-learn`, known as `sklearn`, is an open-source, robust library for machine learning in Python.
* It is created to streamline the process of implementing machine learning and statistical models in Python.
* The package comes with standard machine learning datasets, and you can import it without downloading them from an external website or database.
* Since we will go over a classification example, we will be using the [`wine dataset`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html#sklearn.datasets.load_wine) (Click it for more information).

In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler 

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier 

from sklearn.metrics import classification_report

In [2]:
# load data
wine_data = load_wine()

<font size='4'>

* Executing the above code returns a dictionary-like object (we just learned!) that contains data and metadata.
    * **Metadata**: This is a terminology for data dictionary, i.e., a description of the data itself.

In [3]:
# Convert the data to a Pandas dataframe
wine_df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)

# Add the target label (add a new column)
wine_df['target'] = wine_data.target

# Preview
wine_df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


### Exploratory Data Analysis

<font size='4'>

* Before conducting any data analysis, always check the quality of the dataset with exploratory data analysis.
* You can call `.info()` method to print out a summary of each column.

In [4]:
wine_df.Info()

AttributeError: 'DataFrame' object has no attribute 'Info'

<font size='4'>

* There are 178 data samples with 14 columns including the target column (output that we would like to predict).
* Luckily, no missing values are in our dataset from `Non-Null Count`.
* All features are `float64` except for target column.
* The dataset consumes 19.6 KB of memory.

In [ ]:
wine_df.describe()

<font size='4'>

* Since `target` is a categorical variable, we check the frequency and proportion using `.value_counts()` method.

In [ ]:
pd.concat([wine_df['target'].value_counts(), wine_df['target'].value_counts(normalize=True)*100], axis=1) 

## Data preprocessing

<font size='4'>

* Preprocessing is important prior to applying machine learning algorithms.
    * Check missing values, outliers, duplicates, errors, and data types.
    * Avoid "Garbage in, garbage out."
* Machine learning models typically require numerical inputs.
* Another practice is to standardize the input (via Z-transform) to make predictors more comparable. 
    * We do not want predictor A has a magnitude of 10000, while predictor B has a magnitude of 0.1.
    * We can achieve the goal using `StandardScaler()` class.
    * Each value goes through the following transformation columnwise, i.e., $x = (x - x_{mean})/x_{sd}$.

In [ ]:
# Remember to import standardscaler (code should be written in the beginning of the Jupyter notebook).
# Split data into features (input) and label (output)
print(wine_data.feature_names)

# Always make a copy to avoid making changes on the raw data (when you have sufficient amount of memory).
x_mat = wine_df[wine_data.feature_names].copy
y_mat = wine_df['target'].copy()

<font size='4'>
<img src="figures/sklearn_flowchart_1.png" alt="drawing" width="900"/>
    
* In this flowchart, no output data are involved. We use `.transform()` in the last step.

In [ ]:
# Instantiate scaler and fit on features
scaler_obj = StandardScaler()
#scaler_obj = sklearn.preprocessing.StandardScaler() (this also works, but lengthy)

# apply changes to training data
# and update parameters (in this case, no model parameters are available, so this is optional)
scaler_obj.fit(x_mat)

# apply changes to any data and assign it with another variable
x_scaled_mat = scaler_obj.transform(x_mat)

# view the transformed output
print(type(x_scaled_mat))
print(x_scaled_mat.shape)
x_scaled_mat[0, :] 

## Model training

<font size='4'>

* As we previously mentioned, we need to have training and testing set when we perform classification tasks.
* If rows of input data are independent of each other, we can randomly select training and testing set.
* Sklearn package has a built-in function called `train_test_split()`.
* We usually set 70% data to train and 30% to test. The exact ratio vary depending on the volume of the data.

In [ ]:
# remember to import train_test_split in the beginning
x_train_scaled, x_test_scaled, y_train, y_test = train_test_split(x_scaled_mat, y_mat, train_size=0.7, random_state=100)
# train_test_split(x_input, y_input, training_proportion, seed_to_make_it_reproducible)

# Check the splits are correct
print(x_train_scaled.shape[0])
print(x_test_scaled.shape[0])

### Model building

<font size='4'>

* Sklearn has numerous built-in classification methods. We will demonstrate a couple of methods including
    * logistic regression
    * support vector machine
    * decision tree classifier
    
* <img src="figures/sklearn_flowchart_2.png" alt="drawing" width="900"/>

    * In this flowchart, both input and output (in the training set) are involved, so `.predict()` is used finally.

In [ ]:
# Iniatiating the models 
# Sometimes, you need to modify the parameters inside each of the function.
# If not, the model will use its default values.
# write one model and the rest two are completed by students
# Initialize an object (will use default parameters if you don't adjust the values):
logistic_obj = LogisticRegression()

# Training the models 
logistic_obj.fit(x_train_scaled, y_train)

# Making predictions with each model
log_reg_preds = logistic_obj.predict(x_test_scaled)
print(log_reg_preds)

In [ ]:
# Support vector machine (SVM)
# Initialize an object (will use default parameters if you don't adjust the values):
svm_obj = SVC(probability=True)

# Training the models 
svm_obj.fit(x_train_scaled, y_train)

# Making predictions with each model
svm_preds = svm_obj.predict(x_test_scaled)
print(svm_preds)

In [ ]:
# Decision tree
# Initialize an object (will use default parameters if you don't adjust the values):
d_tree_obj = DecisionTreeClassifier()

# Training the models 
d_tree_obj.fit(x_train_scaled, y_train)

# Making predictions with each model
d_tree_preds = d_tree_obj.predict(x_test_scaled)
print(d_tree_preds)

In [ ]:
y_preds_three_method = np.stack([log_reg_preds, svm_preds, d_tree_preds], axis=0).T
# Or y_preds_three_method = np.stack([log_reg_preds, svm_preds, d_tree_preds], axis=1)
print(y_preds_three_method[:5,:])

In [ ]:
# You can view the probability vector per measure per method.
log_reg_probs = logistic_obj.predict_proba(x_test_scaled)
svm_probs = svm_obj.predict_proba(x_test_scaled)
tree_probs = d_tree_obj.predict_proba(x_test_scaled)

print(log_reg_probs[:5,:])
print(svm_probs[:5,:])
print(d_tree_probs[:5,:]) # prints rows 0-4 (the first 5)

## Model Evaluation

<font size='4'>

* In our case, we will use `classification_report()` to build a text report showing main classification metrics such as `precision`, `recall`, `f1_score`, `accuracy`, etc.

In [ ]:
# from sklearn.metrics import classification_report (put it in the beginning- essential when writing things as python scripts)
# Store model predictions in a dictionary
# this makes it easier to iterate through each model
# and print the results. 
model_preds = {
    'Logistic Regression': log_reg_preds,
    'Support Vector Machine': svm_preds,
    'Decision Tree': d_tree_preds
}

for model, preds in model_preds.items():
    print('{} Results: \n {}\n'format(model, classification_report(y_test, preds, output_dict=True)))

In [ ]:
for model, preds in model_preds.items():
    print('{} Results: \n {}\n'format(model, classification_report(y_test, preds, output_dict=False))) 

# Look at accuracy, unweighted average or weighted average to determine which model performs the best

<font size='4'>
The reported averages include 

* (sample) average (only for multilabel classification). 
* macro average (averaging the unweighted mean per label), 
* weighted average (averaging the support-weighted mean per label).

Which method performs the best?